# 08_物件偵測_mediapipe.py
**電腦視覺與深度學習 › Day18 CNN + Teachable Machine + MediaPipe**

前面的分類只回答「整張圖是什麼」,這支程式更進一步:找出畫面裡每個東西在哪裡,畫上框框,一次可以抓很多個。它是 Day20 YOLO 的前導。

> 🎧 有逐段旁白、逐句拆解的版本在 [pyl Academy](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/4/5)。
>
> **怎麼用**:由上往下,每格按左邊 ▶ 執行;或選單「執行階段 › 全部執行」。
>
> 📷 Colab 沒有攝影機、也開不了視窗:下面會**改用範例圖片**代替鏡頭,影像直接顯示在格子底下(只顯示前幾張)。

In [ ]:
#@title 0. 準備環境(下載課程範例檔、安裝套件)
!git clone -q --depth 1 https://github.com/friencdking0000/2026AI-STUST.git /content/2026AI-STUST 2>/dev/null; echo 課程檔案已就緒
%cd "/content/2026AI-STUST/Week2/DAY9"
!pip -q install mediapipe
import sys, os; sys.argv = sys.argv[:1]  # 讓 argparse / sys.argv 在 notebook 裡也正常
__file__ = os.path.join(os.getcwd(), '08_物件偵測_mediapipe.py')  # notebook 沒有 __file__,程式靠它找同資料夾的檔案

In [ ]:
#@title 1. Colab 沒有攝影機和視窗:改用範例圖片、影像直接顯示在下面
import cv2, numpy as np, urllib.request, os
from google.colab.patches import cv2_imshow
def _pyl_sample():
    if not os.path.exists('/content/_sample.jpg'):
        urllib.request.urlretrieve('https://ultralytics.com/images/bus.jpg', '/content/_sample.jpg')
    return cv2.imread('/content/_sample.jpg')
class _PylCam:
    def __init__(self, *a, **k): self.n = 0; self.img = _pyl_sample()
    def isOpened(self): return True
    def read(self):
        self.n += 1
        return (True, self.img.copy()) if self.n <= 30 else (False, None)
    def set(self, *a): return True
    def get(self, *a): return 0
    def release(self): pass
_pyl_VC = cv2.VideoCapture
cv2.VideoCapture = lambda src=0, *a, **k: _PylCam() if isinstance(src, int) else _pyl_VC(src, *a, **k)
_pyl_shown = [0]
def _pyl_imshow(name, img):
    _pyl_shown[0] += 1
    if _pyl_shown[0] <= 6:
        print('▼', name); cv2_imshow(img)
cv2.imshow = _pyl_imshow
cv2.waitKey = lambda *a: ord('q') if _pyl_shown[0] >= 6 else -1
cv2.destroyAllWindows = lambda *a: None
cv2.namedWindow = lambda *a, **k: None
cv2.createTrackbar = lambda *a, **k: None
cv2.getTrackbarPos = lambda *a, **k: 0
cv2.setMouseCallback = lambda *a, **k: None
cv2.resizeWindow = cv2.moveWindow = cv2.setWindowProperty = cv2.setWindowTitle = lambda *a, **k: None
_pyl_imread = cv2.imread
def _pyl_imread2(path, *a, **k):
    img = _pyl_imread(path, *a, **k)
    if img is None and not os.path.exists(str(path)):
        print('(找不到', path, ':課程 repo 沒附這張圖,改用範例圖片)')
        img = _pyl_sample()
    return img
cv2.imread = _pyl_imread2
print('已改用範例圖片代替攝影機')

### 分類和偵測的差別
註解說明:這裡用 Google 推薦的 EfficientDet-Lite0 模型,在 COCO 資料集上訓練過,認得 80 類東西。

In [ ]:
# 08 — MediaPipe 物件偵測 (Object Detection)

### 匯入
匯入 MediaPipe 和它的 tasks 介面,以及畫中文用的 Pillow。

In [ ]:
#
# 相比 05_遷移學習_MobileNet 只回答「這張圖是什麼類別」，
# Object Detection 進階回答「東西在哪裡」+ 畫框框 + 一次抓多個。
#
# 用 Google 官方推薦的 EfficientDet-Lite0 模型（COCO 資料集，80 類），
# 320x320 輸入，第一次執行會自動下載 4.5MB 的 .tflite 檔。
#
# 需要：pip install mediapipe opencv-python pillow
# 若沒裝：先 pip install -r requirements.txt

import os
import time
import urllib.request
from pathlib import Path

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

### 第一次自動下載模型
如果資料夾裡還沒有模型檔,就用 urlretrieve 從 Google 下載大約 4.5 MB 的 tflite 檔。

In [ ]:
from PIL import Image, ImageDraw, ImageFont

# ====== 模型下載 ======
BASE = Path(__file__).parent
MODEL_PATH = BASE / "efficientdet_lite0.tflite"
MODEL_URL  = ("https://storage.googleapis.com/mediapipe-models/"
              "object_detector/efficientdet_lite0/int8/latest/efficientdet_lite0.tflite")

### 中文翻譯表
COCO 的 80 個英文類別名稱,這裡做了一張中文對照表,沒翻到的就顯示英文。

In [ ]:
if not MODEL_PATH.exists():
    print(f"第一次執行：下載 EfficientDet-Lite0 到 {MODEL_PATH} (~4.5MB) ...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("下載完成！")

# ====== 中文字型 ======
FONT_CANDIDATES = [
    "C:/Windows/Fonts/msjh.ttc",
    "C:/Windows/Fonts/msyh.ttc",
    "C:/Windows/Fonts/simhei.ttf",
]
FONT_PATH = next((p for p in FONT_CANDIDATES if os.path.exists(p)), None)

# COCO 80 類的中文翻譯（沒翻的顯示英文原文）
COCO_ZH = {
    "person":"人", "bicycle":"腳踏車", "car":"汽車", "motorcycle":"機車",
    "airplane":"飛機", "bus":"公車", "train":"火車", "truck":"卡車",
    "boat":"船", "traffic light":"紅綠燈", "fire hydrant":"消防栓",
    "stop sign":"停止標誌", "parking meter":"停車計時器", "bench":"長椅",
    "bird":"鳥", "cat":"貓", "dog":"狗", "horse":"馬", "sheep":"羊",
    "cow":"牛", "elephant":"大象", "bear":"熊", "zebra":"斑馬",
    "giraffe":"長頸鹿", "backpack":"背包", "umbrella":"傘",
    "handbag":"手提包", "tie":"領帶", "suitcase":"行李箱", "frisbee":"飛盤",
    "skis":"滑雪板", "snowboard":"雪橇", "sports ball":"球", "kite":"風箏",
    "baseball bat":"球棒", "baseball glove":"棒球手套", "skateboard":"滑板",
    "surfboard":"衝浪板", "tennis racket":"網球拍", "bottle":"瓶子",
    "wine glass":"酒杯", "cup":"杯子", "fork":"叉子", "knife":"刀",
    "spoon":"湯匙", "bowl":"碗", "banana":"香蕉", "apple":"蘋果",
    "sandwich":"三明治", "orange":"橘子", "broccoli":"青花菜",
    "carrot":"紅蘿蔔", "hot dog":"熱狗", "pizza":"披薩", "donut":"甜甜圈",
    "cake":"蛋糕", "chair":"椅子", "couch":"沙發", "potted plant":"盆栽",
    "bed":"床", "dining table":"餐桌", "toilet":"馬桶", "tv":"電視",
    "laptop":"筆電", "mouse":"滑鼠", "remote":"遙控器", "keyboard":"鍵盤",
    "cell phone":"手機", "microwave":"微波爐", "oven":"烤箱",
    "toaster":"烤麵包機", "sink":"水槽", "refrigerator":"冰箱", "book":"書",
    "clock":"時鐘", "vase":"花瓶", "scissors":"剪刀", "teddy bear":"泰迪熊",
    "hair drier":"吹風機", "toothbrush":"牙刷",
}

### ⚠️ 依類別配色
用類別名稱的雜湊值決定顏色,同一類就同一個顏色。要注意:Python 每次執行時字串的雜湊值都不一樣,所以只有在同一次執行裡顏色才固定,重開程式顏色就換了。

In [ ]:
def 依類別配色(name):
    """相同類別給相同顏色，用 hash 分配（BGR）"""
    h = hash(name) & 0xFFFFFF
    return (h & 0xFF, (h >> 8) & 0xFF, (h >> 16) & 0xFF)

### 彩色底加白字的標籤
先量出文字的寬高,畫一個彩色底的方塊,再在上面寫白字。

### 建立偵測器
設定模型檔、影片模式、最多回傳 10 個框,以及信心低於 0.4 的框丟掉。

In [ ]:
def 畫中文標籤(img_bgr, text, xy, size, bg_bgr):
    """在指定位置畫「彩色底 + 白字」"""
    img_pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img_pil)
    font = ImageFont.truetype(FONT_PATH, size)
    l, t, r, b = draw.textbbox((0, 0), text, font=font)
    tw, th = r - l, b - t
    x, y = xy
    # PIL 要 RGB，把 BGR 換過來
    bg_rgb = (bg_bgr[2], bg_bgr[1], bg_bgr[0])
    draw.rectangle((x, y, x + tw + 6, y + th + 6), fill=bg_rgb)
    draw.text((x + 3, y + 3), text, font=font, fill=(255, 255, 255))
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)


# ====== 建立 detector ======
options = vision.ObjectDetectorOptions(
    base_options=python.BaseOptions(model_asset_path=str(MODEL_PATH)),
    running_mode=vision.RunningMode.VIDEO,   # 影片模式，同步處理
    max_results=10,                          # 一次最多回 10 個框
    score_threshold=0.4,                     # 信心低於 0.4 的丟掉
)

### 開鏡頭
打開鏡頭,準備計時和 FPS 的紀錄。

In [ ]:
detector = vision.ObjectDetector.create_from_options(options)

# ====== 開 webcam ======
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise RuntimeError("無法開啟 webcam")

print("按 q 離開；把物體對著鏡頭讓它畫框")

start_ts = time.time()
fps_hist = []

### 送進 MediaPipe 偵測
MediaPipe 要 RGB 格式,所以先轉色、包成 mp.Image。影片模式需要一個一直增加的毫秒時間戳,再呼叫 detect_for_video。

### 畫框與標籤
每個偵測結果都有一個框:左上角座標和寬高,還有類別和分數。用 rectangle 畫框,上方貼中文標籤和百分比。

### FPS 與收拾
最近 30 格的 FPS 取平均,寫在左上角。按 q 離開,最後放掉鏡頭、關閉偵測器。

In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_start = time.time()

    # MediaPipe 需要 RGB numpy 影像
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    # VIDEO 模式需要單調遞增的時間戳（毫秒）
    ts_ms = int((time.time() - start_ts) * 1000)
    result = detector.detect_for_video(mp_image, ts_ms)

    # 畫每個偵測到的物件
    for det in result.detections:
        bbox = det.bounding_box
        cat  = det.categories[0]
        x, y = bbox.origin_x, bbox.origin_y
        w, h = bbox.width, bbox.height

        name_zh = COCO_ZH.get(cat.category_name, cat.category_name)
        color   = 依類別配色(cat.category_name)

        # 框
        cv2.rectangle(frame, (x, y), (x + w, y + h), color, 3)
        # 上方彩色標籤
        frame = 畫中文標籤(frame, f"{name_zh} {cat.score*100:.0f}%",
                          (x, max(0, y - 30)), 20, color)

    # FPS
    fps_hist.append(1.0 / max(time.time() - frame_start, 1e-3))
    if len(fps_hist) > 30:
        fps_hist.pop(0)
    fps = sum(fps_hist) / len(fps_hist)
    frame = 畫中文標籤(frame, f"FPS {fps:.1f}", (10, 10), 22, (30, 30, 30))

    cv2.imshow("MediaPipe Object Detection (COCO 80 類)", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
detector.close()

## 小結
你從分類走到了偵測:不只知道是什麼,還知道在哪裡。Day20 的 YOLO 會讓你自己訓練要偵測的類別。

👉 回到 [pyl Academy 這堂課](https://friencdking0000.github.io/pyl-academy/#/learn/PY-CVAI/4/5) 看逐句拆解、做練習。